In [0]:
# ============================================================
# 04_silver_unified
# ============================================================
# Purpose : UNION chicago_clean + dallas_clean into one conformed table. Gold reads ONLY from here.

# Why this is necessary:
#   Without unified Silver, Gold notebooks contain
#   Adding a 3rd city forces changes to every Gold notebook. With this notebook, Gold is fully city-agnostic.

# Reads  : silver.chicago_clean
#          silver.dallas_clean
#          silver.chicago_violations
#          silver.dallas_violations

# Writes : silver.inspections_unified 
#          silver.violations_unified
#
# Write mode: OVERWRITE (consistent with Silver-1 notebooks)
# ============================================================


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

CATALOG = "final_project_databi_group8"

CHI_CLEAN    = f"{CATALOG}.silver.chicago_clean"
DAL_CLEAN    = f"{CATALOG}.silver.dallas_clean"
CHI_VIOLS    = f"{CATALOG}.silver.chicago_violations"
DAL_VIOLS    = f"{CATALOG}.silver.dallas_violations"
UNIFIED_INSP = f"{CATALOG}.silver.inspections_unified"
UNIFIED_VIOLS= f"{CATALOG}.silver.violations_unified"

print(f"Catalog  : {CATALOG}")
print(f"Sources  : {CHI_CLEAN}")
print(f"           {DAL_CLEAN}")
print(f"Targets  : {UNIFIED_INSP}")
print(f"           {UNIFIED_VIOLS}")

In [0]:
# Verify all Silver-1 tables exist
for tbl in [CHI_CLEAN, DAL_CLEAN, CHI_VIOLS, DAL_VIOLS]:
    try:
        spark.table(tbl).limit(1).collect()
        print(f"Found: {tbl}")
    except Exception as e:
        raise Exception(
            f"Table not found: {tbl}\n"
            "Run 02_silver_chicago and 03_silver_dallas first.\n"
            f"Error: {e}"
        )


In [0]:
# Define unified schema These columns appear in inspections_unified.
# Chicago provides: aka_name, license_number, facility_type,
#                   risk_category, inspection_result
# Dallas provides:  inspection_score (numeric)
# Both provide:     everything else
# Missing columns were added as NULL in Silver-1 notebooks.

UNIFIED_COLS = [
    "inspection_id",
    "restaurant_name",
    "aka_name",
    "license_number",
    "facility_type",
    "risk_category",
    "street_address",
    "city",
    "state",
    "zip_code",
    "latitude",
    "longitude",
    "inspection_date",
    "inspection_type",
    "inspection_result",
    "inspection_score",
    "violation_count",
    "city_source",
    "_batch_id",
    "_bronze_extract_ts",
    "_silver_ts",
    "_silver_date",
    "_silver_version",
    "_dqx_status",
]

TYPE_MAP = {
    "violation_count":    IntegerType(),
    "inspection_score":   DoubleType(),
    "latitude":           DoubleType(),
    "longitude":          DoubleType(),
}

print(f"Unified schema: {len(UNIFIED_COLS)} columns")
for c in UNIFIED_COLS:
    print(f"  {c}")


In [0]:
# Select Chicago columns
chi_available = spark.table(CHI_CLEAN).columns
chi_missing   = [c for c in UNIFIED_COLS if c not in chi_available]

if chi_missing:
    print(f"missing from chicago_clean: {chi_missing}")
    print("Adding as NULL")

df_chi = spark.table(CHI_CLEAN).select(
    [c for c in UNIFIED_COLS if c in chi_available]
)
for col in chi_missing:
    df_chi = df_chi.withColumn(col, F.lit(None).cast(TYPE_MAP.get(col, "string")))

df_chi = df_chi.select(UNIFIED_COLS)

print(f"Chicago rows   : {df_chi.count():,}")
print(f"Chicago columns: {len(df_chi.columns)}")

# Select Dallas columns
dal_available = spark.table(DAL_CLEAN).columns
dal_missing   = [c for c in UNIFIED_COLS if c not in dal_available]

if dal_missing:
    print(f"missing from dallas_clean: {dal_missing}")

df_dal = spark.table(DAL_CLEAN).select(
    [c for c in UNIFIED_COLS if c in dal_available]
)
for col in dal_missing:
    df_dal = df_dal.withColumn(col, F.lit(None).cast(TYPE_MAP.get(col, "string")))

df_dal = df_dal.select(UNIFIED_COLS)

print(f"Dallas rows    : {df_dal.count():,}")
print(f"Dallas columns : {len(df_dal.columns)}")

In [0]:
#Schema alignment check before UNION
chi_schema = {f.name: f.dataType for f in df_chi.schema}
dal_schema = {f.name: f.dataType for f in df_dal.schema}

mismatches = [
    (col, str(chi_schema[col]), str(dal_schema[col]))
    for col in UNIFIED_COLS
    if col in chi_schema and col in dal_schema
    and chi_schema[col] != dal_schema[col]
]

if mismatches:
    print("Type mismatches — casting Dallas to match Chicago:")
    for col, ct, dt in mismatches:
        print(f"  {col}: CHI={ct}, DAL={dt} casting DAL")
        df_dal = df_dal.withColumn(col, F.col(col).cast(chi_schema[col]))
else:
    print("All column types match - UNION ALL is safe")

# UNION ALL
df_unified = (
    df_chi
    .unionAll(df_dal)
    .withColumn("_merged_ts",   F.current_timestamp())
    .withColumn("_merged_date", F.current_date().cast("string"))
)

chi_n  = df_chi.count()
dal_n  = df_dal.count()
total  = df_unified.count()

print(f"\nChicago rows   : {chi_n:,}")
print(f"Dallas rows    : {dal_n:,}")
print(f"Unified total  : {total:,}")
print(f"Expected total : {chi_n + dal_n:,}")
assert total == chi_n + dal_n, "Row count mismatch after UNION!"
print("Row count validated")


In [0]:
# Build unified violations table
VIOL_COLS = [
    "inspection_id",
    "violation_code",
    "violation_description",
    "violation_points",      # NULL for Chicago (no per-violation points)
    "violation_detail",      # NULL for Chicago (no regulatory text column)
    "inspector_comment",
    "violation_slot",        # NULL for Chicago (not wide-format)
    "city_source",
    "_batch_id",
]

VIOL_TYPE_MAP = {
    "violation_points": DoubleType(),
    "violation_slot":   IntegerType(),
}

def align_viol_table(tbl_name, cols):
    df = spark.table(tbl_name)
    existing = df.columns
    for c in cols:
        if c not in existing:
            df = df.withColumn(c, F.lit(None).cast(VIOL_TYPE_MAP.get(c, "string")))
    return df.select(cols)

df_chi_viols = align_viol_table(CHI_VIOLS, VIOL_COLS)
df_dal_viols = align_viol_table(DAL_VIOLS, VIOL_COLS)

df_viols_unified = (
    df_chi_viols
    .unionAll(df_dal_viols)
    .withColumn("_merged_ts",   F.current_timestamp())
    .withColumn("_merged_date", F.current_date().cast("string"))
)

print(f"Chicago violations : {df_chi_viols.count():,}")
print(f"Dallas violations  : {df_dal_viols.count():,}")
print(f"Unified violations : {df_viols_unified.count():,}")


In [0]:
# Write unified tables
(
    df_unified
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("city_source")
    .saveAsTable(UNIFIED_INSP)
)
print(f" {UNIFIED_INSP}")
print(f"  Rows    : {df_unified.count():,}")
print(f"  Columns : {len(df_unified.columns)}")

(
    df_viols_unified
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("city_source")
    .saveAsTable(UNIFIED_VIOLS)
)
print(f" {UNIFIED_VIOLS}")
print(f"  Rows    : {df_viols_unified.count():,}")
print(f"  Columns : {len(df_viols_unified.columns)}")

print("\n Gold layer must read from:")
print(f"  {UNIFIED_INSP}")
print(f"  {UNIFIED_VIOLS}")


In [0]:
# Verification
print("=== inspections_unified by city ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                              AS total_inspections,
        ROUND(AVG(inspection_score), 2)       AS avg_score,
        ROUND(AVG(violation_count), 2)        AS avg_violations,
        COUNT(DISTINCT risk_category)         AS risk_categories,
        COUNT(DISTINCT inspection_type)       AS inspection_types,
        MIN(inspection_date)                  AS earliest_date,
        MAX(inspection_date)                  AS latest_date
    FROM {UNIFIED_INSP}
    GROUP BY city_source
    ORDER BY city_source
"""))

print("\n=== violations_unified by city ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                              AS total_violation_rows,
        COUNT(DISTINCT violation_code)        AS unique_codes,
        SUM(CASE WHEN violation_points IS NULL THEN 1 ELSE 0 END) AS null_points_chi,
        SUM(CASE WHEN violation_slot   IS NULL THEN 1 ELSE 0 END) AS null_slot_chi
    FROM {UNIFIED_VIOLS}
    GROUP BY city_source
    ORDER BY city_source
"""))
